# Fine-Tune VitPose on Harmony4D Dataset

This notebook fine-tunes a VitPose model on the Harmony4D dataset for pose estimation.

# 0. Setup Environment and Dependencies

In [ ]:
# Select the device to use
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Install required packages
!pip -q install datasets evaluate transformers wandb huggingface_hub supervision

In [ ]:
# Clone VitPose repository
!git clone https://github.com/ViTAE-Transformer/ViTPose.git
%cd ViTPose
!pip install -r requirements.txt
!pip install -e .
%cd ..

In [ ]:
# Login to Hugging Face and W&B to track model training
!huggingface-cli login
!wandb login

## 0.1 Track run results on W&B

In [ ]:
import wandb
import random

# Start a new wandb run to track this script
wandb.init(
    # Set the project where this run will be logged
    project="jiu-jitsu-vitpose",

    # Track hyperparameters and run metadata
    config={
        "learning_rate": 0.0001,
        "architecture": "VitPose",
        "dataset": "Harmony4D",
        "epochs": 10,
    }
)

# 1 Prepare Dataset

## 1.1 Download Harmony4D

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile
import os
import shutil
from google.colab import drive
drive.mount('/content/drive')

REPO_ID = "Jyun-Ting/Harmony4D"
TRAIN_FILENAME = "train/01_hugging.zip"
FOLDER_NAME = "datasets/"

# Download the dataset
train_path = hf_hub_download(repo_id=REPO_ID, filename=TRAIN_FILENAME, repo_type="dataset")
print(f"Downloaded training data to {train_path}")

# If the dataset isn't already on your drive, copy it
if not os.path.exists("/content/drive/MyDrive/" + FOLDER_NAME + TRAIN_FILENAME):
    os.makedirs("/content/drive/MyDrive/" + FOLDER_NAME, exist_ok=True)
    shutil.copy(train_path, "/content/drive/MyDrive/" + FOLDER_NAME)

## 1.2 Extract and Prepare Dataset

For VitPose, we need to extract keypoint annotations from the Harmony4D dataset.

In [ ]:
# Extract the dataset
import zipfile
import json
import glob
from PIL import Image
import numpy as np
from datasets import Dataset, Features, Value, Array2D, Array3D, Sequence

# Extract zip file
extract_dir = "./harmony4d_extracted"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(train_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

## 1.3 Create Custom Dataset

We need to convert the Harmony4D dataset format to a format suitable for pose estimation.

In [ ]:
# Function to process the dataset and create a format compatible with VitPose
def process_harmony4d_for_pose():
    data_dict = {
        "image_path": [],
        "image": [],
        "keypoints": [],
        "bbox": []
    }

    # Find all annotation JSON files
    annotation_files = glob.glob(f"{extract_dir}/**/*.json", recursive=True)

    # Process a smaller subset first to ensure everything works
    # This helps avoid RAM issues
    MAX_SAMPLES = 1000  # Increase this as needed
    processed_count = 0

    for anno_file in annotation_files:
        if processed_count >= MAX_SAMPLES:
            break

        try:
            # Load annotation file
            with open(anno_file, 'r') as f:
                annotations = json.load(f)

            # Get corresponding image path
            image_path = anno_file.replace('.json', '.jpg')
            if not os.path.exists(image_path):
                continue

            # Load image to verify it works
            img = Image.open(image_path)
            img_width, img_height = img.size

            # Process each person in the annotation
            for person in annotations.get('annotations', []):
                if 'keypoints' not in person:
                    continue

                keypoints = np.array(person['keypoints']).reshape(-1, 3)  # x, y, visibility
                bbox = person.get('bbox', [0, 0, img_width, img_height])  # x, y, width, height

                # Store data
                data_dict["image_path"].append(image_path)
                data_dict["image"].append(img)
                data_dict["keypoints"].append(keypoints)
                data_dict["bbox"].append(bbox)

                processed_count += 1
                if processed_count >= MAX_SAMPLES:
                    break

        except Exception as e:
            print(f"Error processing {anno_file}: {e}")
            continue

    print(f"Processed {processed_count} pose instances from {len(annotation_files)} annotation files")
    return data_dict

# Process the dataset
print("Processing Harmony4D dataset...")
dataset_dict = process_harmony4d_for_pose()

# Convert to HuggingFace Dataset
pose_dataset = Dataset.from_dict(dataset_dict)

# Split dataset into train and validation
pose_dataset = pose_dataset.train_test_split(test_size=0.1)
train_dataset = pose_dataset["train"]
eval_dataset = pose_dataset["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")

# 2 Fine-tuning VitPose

## 2.1 Load Pre-trained Model

In [ ]:
from transformers import AutoProcessor, VitPoseForPoseEstimation

# Load the model and processor
processor = AutoProcessor.from_pretrained("usyd-community/vitpose-base-simple")
model = VitPoseForPoseEstimation.from_pretrained("usyd-community/vitpose-base-simple")
model.to(device)

## 2.2 Define Data Processing Functions

In [ ]:
def preprocess_pose_data(examples):
    # Process images and keypoints
    images = examples["image"]
    keypoints = examples["keypoints"]
    bboxes = examples["bbox"]

    # Prepare inputs for the model
    inputs = processor(
        images=images,
        keypoints=[kp[:, :2] for kp in keypoints],  # Extract x, y
        keypoint_visibility=[kp[:, 2] for kp in keypoints],  # Extract visibility
        boxes=bboxes,
        return_tensors="pt"
    )

    return inputs

## 2.3 Memory Efficient Data Processing

To avoid RAM issues, we'll process the dataset in smaller chunks.

In [ ]:
# Define a custom data collator for batching
def custom_data_collator(features):
    batch = {}

    # Collect all the keys
    keys = features[0].keys()

    # Process each key
    for key in keys:
        if key == "pixel_values":
            # Stack pixel values
            batch[key] = torch.stack([feature[key] for feature in features])
        elif key == "keypoints":
            # Process keypoints - may need adjustment based on exact format
            batch[key] = [feature[key] for feature in features]
        elif key == "labels" or key == "heatmaps":
            # Stack labels
            batch[key] = torch.stack([feature[key] for feature in features]) if all(key in feature for feature in features) else None
        else:
            # For other keys
            batch[key] = [feature[key] for feature in features] if all(key in feature for feature in features) else None

    return batch

## 2.4 Define Training Arguments

In [ ]:
from transformers import TrainingArguments

# Configure training arguments with memory optimization
training_args = TrainingArguments(
    output_dir="./vitpose-harmony4d",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,  # Reduce batch size to save memory
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # Accumulate gradients to simulate larger batch size
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    push_to_hub=True,
    fp16=True,  # Use mixed precision training to save memory
    dataloader_num_workers=2,
    logging_dir="./logs",
    logging_steps=10,
    report_to="wandb",
    save_total_limit=2,  # Keep only the 2 best checkpoints to save disk space
)

## 2.5 Define Metrics

In [ ]:
import numpy as np
import evaluate

def compute_pose_metrics(eval_pred):
    predictions, labels = eval_pred
    metrics = {}

    # Calculate Mean Per Joint Position Error (MPJPE)
    pred_keypoints = predictions.keypoints  # Adjust based on actual output format
    true_keypoints = labels.keypoints

    if pred_keypoints is not None and true_keypoints is not None:
        # Calculate Euclidean distance for each joint
        distances = np.sqrt(np.sum((pred_keypoints - true_keypoints) ** 2, axis=-1))
        metrics["mpjpe"] = np.mean(distances)

        # Calculate PCK (Percentage of Correct Keypoints)
        threshold = 0.05  # 5% of the person bounding box diagonal
        pck = np.mean(distances < threshold)
        metrics["pck"] = pck

    return metrics

## 2.6 Process Datasets for Training

In [ ]:
# Process datasets - do this in batches to save memory
print("Processing training dataset...")
processed_train_dataset = train_dataset.map(
    preprocess_pose_data,
    batched=True,
    batch_size=16,  # Process in small batches
    remove_columns=train_dataset.column_names
)

print("Processing validation dataset...")
processed_eval_dataset = eval_dataset.map(
    preprocess_pose_data,
    batched=True,
    batch_size=16,
    remove_columns=eval_dataset.column_names
)

## 2.7 Create Trainer and Start Training

In [ ]:
from transformers import Trainer

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_train_dataset,
    eval_dataset=processed_eval_dataset,
    data_collator=custom_data_collator,
    compute_metrics=compute_pose_metrics,
)

# Start training
print("Starting training...")
trainer.train()

# 3 Save and Upload Model

In [ ]:
# Save model locally
model_save_path = "./vitpose-harmony4d-finetuned"
trainer.save_model(model_save_path)
print(f"Model saved to {model_save_path}")

# Upload to Hugging Face Hub
trainer.push_to_hub()
print("Model uploaded to Hugging Face Hub")

# 4 Test Fine-tuned Model on Sample Image

In [ ]:
import matplotlib.pyplot as plt
import supervision as sv

# Load a test image
from PIL import Image
import requests

# You can replace this URL with a path to a local image
url = "http://images.cocodataset.org/val2017/000000000139.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# Load our fine-tuned model
finetuned_processor = AutoProcessor.from_pretrained(training_args.output_dir)
finetuned_model = VitPoseForPoseEstimation.from_pretrained(training_args.output_dir)
finetuned_model.to(device)

# Detect humans using Deformable DETR (as in your demo)
from transformers import AutoImageProcessor, DeformableDetrForObjectDetection

detr_processor = AutoImageProcessor.from_pretrained("SenseTime/deformable-detr")
detr_model = DeformableDetrForObjectDetection.from_pretrained("SenseTime/deformable-detr")
detr_model.to(device)

inputs = detr_processor(images=image, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = detr_model(**inputs)

# Process results
target_sizes = torch.tensor([image.size[::-1]])
results = detr_processor.post_process_object_detection(
    outputs, target_sizes=target_sizes, threshold=0.3
)

result = results[0]

# Get the boxes around persons only
person_boxes = result["boxes"][result["labels"] == 1]
person_boxes = person_boxes.cpu().numpy()

# Convert boxes from (x1, y1, x2, y2) to (x1, y1, w, h) format
person_boxes_coco = person_boxes.copy()
person_boxes_coco[:, 2] = person_boxes_coco[:, 2] - person_boxes_coco[:, 0]
person_boxes_coco[:, 3] = person_boxes_coco[:, 3] - person_boxes_coco[:, 1]

# Apply VitPose to each detected person
if len(person_boxes_coco) > 0:
    image_inputs = finetuned_processor(image, boxes=[person_boxes_coco], return_tensors="pt")
    image_inputs = {k: v.to(device) for k, v in image_inputs.items()}

    with torch.no_grad():
        outputs = finetuned_model(**image_inputs)

    pose_results = finetuned_processor.post_process_pose_estimation(outputs, boxes=[person_boxes_coco])
    image_pose_result = pose_results[0]  # results for first image

    # Visualize results
    xy = torch.stack([pose_result['keypoints'] for pose_result in image_pose_result]).cpu().numpy()
    scores = torch.stack([pose_result['scores'] for pose_result in image_pose_result]).cpu().numpy()

    key_points = sv.KeyPoints(
        xy=xy, confidence=scores
    )

    edge_annotator = sv.EdgeAnnotator(
        color=sv.Color.GREEN,
        thickness=1
    )
    vertex_annotator = sv.VertexAnnotator(
        color=sv.Color.RED,
        radius=2
    )
    annotated_frame = edge_annotator.annotate(
        scene=image.copy(),
        key_points=key_points
    )
    annotated_frame = vertex_annotator.annotate(
        scene=annotated_frame,
        key_points=key_points
    )

    # Display the annotated image
    plt.figure(figsize=(10, 10))
    plt.imshow(annotated_frame)
    plt.axis('off')

    # Display boxes
    for box in person_boxes_coco:
        x, y, w, h = box
        rect = plt.Rectangle((x, y), w, h, fill=False, edgecolor='yellow', linewidth=2)
        plt.gca().add_patch(rect)

    plt.show()
else:
    print("No humans detected in the image")

# End W&B session
wandb.finish()